# Telegram Sticker Booth (Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TanYuanXiangElroy/telegram-sticker-booth/blob/main/notebook/sticker_booth.ipynb)

Photo + prompt → Gemini (optional, paid) → green-screen cleanup → 512×512 PNG/WebP → optional ZIP download.

**Defaults are offline and free:** `USE_API = False` and `UPLOAD = False`. Turn them on only when you intentionally want paid generation or Telegram upload.

If you opened an older Google Drive copy of this notebook, prefer this GitHub-backed Colab link (or replace the Drive file with the version from the repo).

## 0. Bootstrap (Colab)

Fresh Colab runtimes do not include this repo. Clone it and install dependencies, then `cd` into the project so imports work.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = Path("/content").exists()
REPO_URL = "https://github.com/TanYuanXiangElroy/telegram-sticker-booth.git"
REPO_DIR = Path("/content/telegram-sticker-booth") if IN_COLAB else (
    Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
)

if IN_COLAB:
    if not (REPO_DIR / "sticker_booth").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
else:
    os.chdir(REPO_DIR)

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)
assert (ROOT / "sticker_booth" / "prompts.py").is_file(), "repo bootstrap failed"
print("bootstrap ok")

## 1. Settings + photo

- **Colab:** run the upload cell and pick a selfie, or set `PHOTO` to a path under the repo.
- **Local:** put a photo in `photos/` (git-ignored) or point `PHOTO` at any image.
- Paid Gemini and Telegram stay **off** unless you flip the flags.

In [ ]:
# --- edit these ---
USE_API = False   # True = paid Gemini call (~US$0.034 / image on gemini-3.1-flash-lite-image)
UPLOAD = False    # True = create/extend a Telegram pack (needs your bot token + user id)
STYLE = "pixel"   # cartoon | cyberpunk | pixel
EMOJI = "😂"      # key from sticker_booth.prompts.EMOJIS
PHOTO = None      # set to a path string, or leave None and upload below on Colab

OUT = ROOT / "notebook" / "out"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
from IPython.display import display
from PIL import Image

photo_path = Path(PHOTO) if PHOTO else None
if photo_path and not photo_path.is_file():
    photo_path = ROOT / PHOTO if PHOTO else None

if photo_path is None or not Path(photo_path).is_file():
    if IN_COLAB:
        from google.colab import files
        print("Upload a selfie…")
        uploaded = files.upload()
        if not uploaded:
            raise SystemExit("No file uploaded")
        name = next(iter(uploaded))
        photo_path = OUT / name
        photo_path.write_bytes(uploaded[name])
    else:
        # offline demo path when no photo: conversion still works from examples/
        photo_path = None
        print("No photo set; offline cells will use examples/sample_raw.png")

if photo_path:
    print("photo:", photo_path)
    display(Image.open(photo_path).resize((256, 256)))

## 2. Credentials (only if USE_API or UPLOAD)

Prefer **Colab Secrets** (key icon) named `GEMINI_API_KEY` / `TELEGRAM_BOT_TOKEN` / `TELEGRAM_USER_ID`.
Otherwise you will be prompted with `getpass` (input is hidden). Keys are never printed.

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v:
                os.environ[name] = v
                return v
        except Exception:
            pass
    # local .env without printing values
    from sticker_booth.env import load_env
    load_env(ROOT / ".env")
    if os.environ.get(name):
        return os.environ[name]
    if (USE_API and name == "GEMINI_API_KEY") or (UPLOAD and name.startswith("TELEGRAM")):
        v = getpass(f"{name} (hidden): ").strip()
        if v:
            os.environ[name] = v
        return v or None
    return None

if USE_API:
    assert load_secret("GEMINI_API_KEY"), "GEMINI_API_KEY required when USE_API=True"
    print("GEMINI_API_KEY: set")
else:
    print("Skipping Gemini key (USE_API=False)")

if UPLOAD:
    assert load_secret("TELEGRAM_BOT_TOKEN"), "TELEGRAM_BOT_TOKEN required when UPLOAD=True"
    assert load_secret("TELEGRAM_USER_ID"), "TELEGRAM_USER_ID required when UPLOAD=True"
    print("Telegram credentials: set")
else:
    print("Skipping Telegram credentials (UPLOAD=False)")

In [ ]:
import numpy as np
from sticker_booth import postprocess as pp
from sticker_booth.prompts import EMOJIS, build_prompt
from sticker_booth.generate import MODEL, COST_PER_IMAGE

def checker(size, n=16):
    y, x = np.indices(size[::-1]) // n
    g = np.where((x + y) % 2, 200, 235).astype(np.uint8)
    return Image.fromarray(np.dstack([g, g, g, np.full_like(g, 255)]), "RGBA")

def show(*images, height=280, gap=12):
    tiles = []
    for im in images:
        im = im.convert("RGBA")
        im = im.resize((round(im.width * height / im.height), height), Image.LANCZOS)
        bg = checker(im.size); bg.alpha_composite(im); tiles.append(bg)
    sheet = Image.new("RGB", (sum(t.width for t in tiles) + gap * (len(tiles) - 1), height), "white")
    x = 0
    for t in tiles:
        sheet.paste(t.convert("RGB"), (x, 0)); x += t.width + gap
    display(sheet)

name = EMOJIS[EMOJI][0]
print(f"{EMOJI} '{name}' style={STYLE} model={MODEL} (~${COST_PER_IMAGE}/img) USE_API={USE_API}")

## 3. Prompt + generate (or load sample)

With `USE_API = False` this loads `examples/sample_raw.png` (no Gemini call).

In [ ]:
prompt = build_prompt(EMOJI, STYLE)
print(prompt.replace(". ", ".\n"))

In [ ]:
raw_path = OUT / f"raw_{name}.png"
if USE_API:
    if not photo_path:
        raise SystemExit("Set PHOTO or upload a selfie before USE_API=True")
    from sticker_booth.generate import gemini, photo_jpeg
    print(f"Calling {MODEL} (paid)…")
    raw_path.write_bytes(gemini(photo_jpeg(photo_path), prompt))
    show(Image.open(photo_path), Image.open(raw_path))
else:
    raw_path = ROOT / "examples" / "sample_raw.png"
    print("Offline: using", raw_path)

raw = Image.open(raw_path).convert("RGB")
print("raw:", raw.size)
show(raw)

## 4. Convert to Telegram sticker + download

In [ ]:
sticker = pp.process(raw_path)
png, webp = pp.save_sticker(sticker, OUT / name)
print(f"{png.name}: {sticker.size} {sticker.mode}, {png.stat().st_size // 1024} KB")
print(f"{webp.name}: {webp.stat().st_size // 1024} KB")
assert sticker.size == (512, 512) and sticker.mode == "RGBA"
show(raw, sticker)

In [ ]:
from sticker_booth.export_zip import write_zip

# export_zip lives on main after the ZIP PR merges; fall back to a tiny local zip if missing
zip_path = OUT / f"{name}_sticker.zip"
try:
    write_zip([(name, png, webp)], zip_path)
except Exception:
    import zipfile
    with zipfile.ZipFile(zip_path, "w") as zf:
        zf.write(png, f"png/{png.name}")
        zf.write(webp, f"webp/{webp.name}")
print("wrote", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(png))
    files.download(str(webp))
    files.download(str(zip_path))
else:
    print("Download from", OUT)

## 5. Optional Telegram upload (opt-in)

Uses the sticker just written under `notebook/out/`, **not** the bundled classic examples. Leave `UPLOAD = False` unless you mean it.

In [ ]:
import json
from sticker_booth.telegram_pack import build_pack

pack_dir = OUT / "pack_one"
pack_dir.mkdir(parents=True, exist_ok=True)
(pack_dir / png.name).write_bytes(png.read_bytes())
(pack_dir / webp.name).write_bytes(webp.read_bytes())
(pack_dir / "pack.json").write_text(json.dumps({
    "stickers": [{"emoji": EMOJI, "png": png.name, "webp": webp.name}]
}, ensure_ascii=False, indent=2), encoding="utf-8")

link = build_pack(
    [pack_dir],
    name="booth_colab",
    title="Booth Colab",
    user_id=os.environ.get("TELEGRAM_USER_ID"),
    chat_id=os.environ.get("TELEGRAM_USER_ID"),
    dry_run=not UPLOAD,
)
print(link)

## Notes

| Mode | What runs |
|---|---|
| Offline (`USE_API=False`) | sample raw → cleanup → PNG/WebP/ZIP download |
| Paid (`USE_API=True`) | your photo → `gemini-3.1-flash-lite-image` → cleanup |
| Upload (`UPLOAD=True`) | Bot API pack from **this run's** `notebook/out` |

Do not commit photos or `.env`. Only make stickers of people who agreed.